# Pipeline 1: Video Summarization Model Training

Train 3 models (RandomForest, XGBoost, BiLSTM) on the **TVSum** dataset to predict frame-level importance scores.

**Dataset:** TVSum50 — 50 YouTube videos with human importance annotations (1-5 scale)

**Feature Extraction:** ResNet18 (pretrained on ImageNet) → 512-dim vectors per 2-sec segment

**Output:** Trained models saved to Google Drive

## Cell 1: Install Dependencies

In [ ]:
!pip install -q torch torchvision xgboost scikit-learn pandas numpy matplotlib scipy yt-dlp opencv-python-headless Pillow

## Cell 2: Mount Google Drive & Set Paths

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os

# All data and models will be saved here
PROJECT_DIR = '/content/drive/MyDrive/ai-video-summarizer'
DATA_DIR = os.path.join(PROJECT_DIR, 'data')
TVSUM_DIR = os.path.join(DATA_DIR, 'raw', 'tvsum')
VIDEO_DIR = os.path.join(TVSUM_DIR, 'videos')
FEATURE_DIR = os.path.join(DATA_DIR, 'processed', 'features')
MODEL_DIR = os.path.join(PROJECT_DIR, 'outputs', 'models')

for d in [VIDEO_DIR, FEATURE_DIR, MODEL_DIR]:
    os.makedirs(d, exist_ok=True)

# Download TVSum annotation files if not present
ANNO_FILE = os.path.join(TVSUM_DIR, 'ydata-tvsum50-anno.tsv')
INFO_FILE = os.path.join(TVSUM_DIR, 'ydata-tvsum50-info.tsv')

print(f'Project dir: {PROJECT_DIR}')
print(f'Anno file exists: {os.path.exists(ANNO_FILE)}')
print(f'Info file exists: {os.path.exists(INFO_FILE)}')

## Cell 3: Download TVSum Videos

**Important:** Upload `ydata-tvsum50-info.tsv` and `ydata-tvsum50-anno.tsv` to `{PROJECT_DIR}/data/raw/tvsum/` before running this cell.

Download from: https://github.com/yalesong/tvsum

In [ ]:
import pandas as pd
import subprocess

if not os.path.exists(INFO_FILE):
    print('ERROR: Upload ydata-tvsum50-info.tsv to', TVSUM_DIR)
else:
    df = pd.read_csv(INFO_FILE, sep='\t')
    
    # Find video ID column
    video_id_col = None
    for col in df.columns:
        if 'video' in col.lower() and 'id' in col.lower():
            video_id_col = col
            break
    
    if video_id_col is None:
        print('Could not find video ID column. Columns:', df.columns.tolist())
    else:
        video_ids = df[video_id_col].dropna().unique()
        print(f'Downloading {len(video_ids)} TVSum videos...')
        
        for vid in video_ids:
            vid = str(vid).strip()
            # Check if already downloaded
            existing = [f for f in os.listdir(VIDEO_DIR) if f.startswith(vid)]
            if existing:
                print(f'[SKIP] {vid} — already exists')
                continue
            
            url = f'https://www.youtube.com/watch?v={vid}'
            output_template = os.path.join(VIDEO_DIR, f'{vid}.%(ext)s')
            try:
                subprocess.run(
                    ['yt-dlp', '-f', 'mp4/best', '-o', output_template, url],
                    check=True, capture_output=True, text=True
                )
                print(f'[OK] {vid}')
            except subprocess.CalledProcessError:
                print(f'[FAIL] {vid} — video may be unavailable')

print(f'\nVideos in {VIDEO_DIR}: {len(os.listdir(VIDEO_DIR))}')

## Cell 4: Extract Frames (2-sec intervals)

In [ ]:
import cv2
import numpy as np

FRAMES_DIR = os.path.join(DATA_DIR, 'processed', 'frames')
SEGMENT_SECONDS = 2

video_files = [f for f in os.listdir(VIDEO_DIR) if f.endswith(('.mp4', '.mkv', '.webm'))]
print(f'Found {len(video_files)} videos to process')

for vf in sorted(video_files):
    video_id = os.path.splitext(vf)[0]
    frames_out = os.path.join(FRAMES_DIR, video_id)
    
    if os.path.exists(frames_out) and len(os.listdir(frames_out)) > 0:
        print(f'[SKIP] {video_id} — frames exist')
        continue
    
    os.makedirs(frames_out, exist_ok=True)
    video_path = os.path.join(VIDEO_DIR, vf)
    cap = cv2.VideoCapture(video_path)
    fps = cap.get(cv2.CAP_PROP_FPS)
    
    if fps <= 0:
        print(f'[FAIL] {video_id} — bad FPS')
        cap.release()
        continue
    
    frame_interval = max(1, int(fps * SEGMENT_SECONDS))
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    seg_idx = 0
    
    while True:
        start = seg_idx * frame_interval
        if start >= total_frames:
            break
        cap.set(cv2.CAP_PROP_POS_FRAMES, start)
        ret, frame = cap.read()
        if not ret:
            break
        path = os.path.join(frames_out, f'frame_{seg_idx:04d}.jpg')
        cv2.imwrite(path, frame)
        seg_idx += 1
    
    cap.release()
    print(f'[OK] {video_id} — {seg_idx} segments')

print('\nFrame extraction complete!')

## Cell 5: Extract ResNet18 Features (GPU Accelerated)

In [ ]:
import torch
import torchvision.models as models
import torchvision.transforms as transforms
from PIL import Image
from tqdm.notebook import tqdm

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')

# Load ResNet18 (remove classification head)
resnet = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
resnet.fc = torch.nn.Identity()
resnet.eval()
resnet.to(DEVICE)

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

# Extract features for each video
video_dirs = sorted([d for d in os.listdir(FRAMES_DIR) if os.path.isdir(os.path.join(FRAMES_DIR, d))])

for video_id in tqdm(video_dirs, desc='Extracting features'):
    output_path = os.path.join(FEATURE_DIR, f'{video_id}.npy')
    if os.path.exists(output_path):
        continue
    
    frames_path = os.path.join(FRAMES_DIR, video_id)
    frame_files = sorted([f for f in os.listdir(frames_path) if f.endswith('.jpg')])
    
    features = []
    for ff in frame_files:
        img = Image.open(os.path.join(frames_path, ff)).convert('RGB')
        tensor = transform(img).unsqueeze(0).to(DEVICE)
        with torch.no_grad():
            feat = resnet(tensor).squeeze().cpu().numpy()
        features.append(feat)
    
    if features:
        np.save(output_path, np.array(features, dtype=np.float32))

print(f'\nFeature files: {len(os.listdir(FEATURE_DIR))}')

## Cell 6: Build Dataset (Features + Annotations)

In [ ]:
import random

RANDOM_SEED = 42
TEST_RATIO = 0.2

# Load annotations
anno_df = pd.read_csv(ANNO_FILE, sep='\t', header=None, names=['video_id', 'category', 'scores'])
annotations = {}
for _, row in anno_df.iterrows():
    vid = str(row['video_id']).strip()
    scores = np.array([float(x) for x in str(row['scores']).split(',') if x != ''], dtype=np.float32)
    annotations[vid] = scores

print(f'Annotations loaded: {len(annotations)} videos')

# Get available videos (have both features + annotations)
available = []
for f in os.listdir(FEATURE_DIR):
    if f.endswith('.npy'):
        vid = f.replace('.npy', '')
        if vid in annotations:
            available.append(vid)
available.sort()
print(f'Available videos: {len(available)}')

# Video-level split
rng = random.Random(RANDOM_SEED)
shuffled = available[:]
rng.shuffle(shuffled)
n_test = max(1, int(len(shuffled) * TEST_RATIO))
test_videos = shuffled[:n_test]
train_videos = shuffled[n_test:]

print(f'Train: {len(train_videos)} videos, Test: {len(test_videos)} videos')


def build_dataset(video_ids, feature_dir, annotations, temporal_radius=1):
    X_list, y_list = [], []
    for vid in video_ids:
        features = np.load(os.path.join(feature_dir, f'{vid}.npy'))
        scores = annotations[vid]
        min_len = min(len(features), len(scores))
        features = features[:min_len]
        scores = scores[:min_len]
        if len(features) == 0:
            continue
        X_list.append(features)
        y_list.append(scores)
    return np.vstack(X_list).astype(np.float32), np.hstack(y_list).astype(np.float32)


def build_temporal_features(features, radius=1):
    num_segments, feat_dim = features.shape
    rows = []
    for i in range(num_segments):
        parts = []
        for offset in range(-radius, radius + 1):
            j = max(0, min(num_segments - 1, i + offset))
            parts.append(features[j])
        rows.append(np.concatenate(parts))
    return np.vstack(rows).astype(np.float32)


def build_temporal_dataset(video_ids, feature_dir, annotations, radius=1):
    X_list, y_list = [], []
    for vid in video_ids:
        features = np.load(os.path.join(feature_dir, f'{vid}.npy'))
        scores = annotations[vid]
        min_len = min(len(features), len(scores))
        features = features[:min_len]
        scores = scores[:min_len]
        if len(features) == 0:
            continue
        temporal = build_temporal_features(features, radius)
        X_list.append(temporal)
        y_list.append(scores)
    return np.vstack(X_list).astype(np.float32), np.hstack(y_list).astype(np.float32)


# Build both raw and temporal datasets
X_train_raw, y_train = build_dataset(train_videos, FEATURE_DIR, annotations)
X_test_raw, y_test = build_dataset(test_videos, FEATURE_DIR, annotations)

X_train_temp, _ = build_temporal_dataset(train_videos, FEATURE_DIR, annotations, radius=1)
X_test_temp, _ = build_temporal_dataset(test_videos, FEATURE_DIR, annotations, radius=1)

print(f'\nRaw — Train: {X_train_raw.shape}, Test: {X_test_raw.shape}')
print(f'Temporal — Train: {X_train_temp.shape}, Test: {X_test_temp.shape}')
print(f'Labels — Train: {y_train.shape}, Test: {y_test.shape}')

## Cell 7: Train RandomForest

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from scipy.stats import spearmanr
import joblib

print('Training Random Forest (512-dim features)...')
rf = RandomForestRegressor(n_estimators=100, random_state=RANDOM_SEED, n_jobs=-1)
rf.fit(X_train_raw, y_train)

rf_preds = rf.predict(X_test_raw)
rf_mse = mean_squared_error(y_test, rf_preds)
rf_mae = mean_absolute_error(y_test, rf_preds)
rf_r2 = r2_score(y_test, rf_preds)
rf_spearman = spearmanr(y_test, rf_preds).correlation

print(f'\nRandom Forest Results:')
print(f'  MSE: {rf_mse:.4f}')
print(f'  MAE: {rf_mae:.4f}')
print(f'  R2:  {rf_r2:.4f}')
print(f'  Spearman: {rf_spearman:.4f}')

rf_path = os.path.join(MODEL_DIR, 'random_forest_video_split.pkl')
joblib.dump(rf, rf_path)
print(f'\nSaved → {rf_path}')

## Cell 8: Train XGBoost (Temporal Context)

In [ ]:
import xgboost as xgb

print('Training XGBoost with temporal context (1536-dim features)...')
xgb_model = xgb.XGBRegressor(
    n_estimators=400,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    objective='reg:squarederror',
    random_state=RANDOM_SEED,
    n_jobs=-1,
)
xgb_model.fit(X_train_temp, y_train)

xgb_preds = xgb_model.predict(X_test_temp)
xgb_mse = mean_squared_error(y_test, xgb_preds)
xgb_mae = mean_absolute_error(y_test, xgb_preds)
xgb_r2 = r2_score(y_test, xgb_preds)
xgb_spearman = spearmanr(y_test, xgb_preds).correlation

print(f'\nXGBoost Results:')
print(f'  MSE: {xgb_mse:.4f}')
print(f'  MAE: {xgb_mae:.4f}')
print(f'  R2:  {xgb_r2:.4f}')
print(f'  Spearman: {xgb_spearman:.4f}')

xgb_path = os.path.join(MODEL_DIR, 'xgboost_video_split_temporal.pkl')
joblib.dump(xgb_model, xgb_path)
print(f'\nSaved → {xgb_path}')

## Cell 9: Train BiLSTM

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

HIDDEN_DIM = 128
NUM_LAYERS = 2
DROPOUT = 0.3
LR = 1e-3
EPOCHS = 60
BATCH_SIZE = 4
PATIENCE = 10


class BiLSTMSummarizer(nn.Module):
    def __init__(self, input_dim=512, hidden_dim=128, num_layers=2, dropout=0.3):
        super().__init__()
        self.lstm = nn.LSTM(input_dim, hidden_dim, num_layers, batch_first=True,
                           bidirectional=True, dropout=dropout if num_layers > 1 else 0.0)
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Sequential(
            nn.Linear(hidden_dim * 2, 64), nn.ReLU(), nn.Dropout(dropout), nn.Linear(64, 1)
        )

    def forward(self, x, lengths=None):
        if lengths is not None:
            packed = nn.utils.rnn.pack_padded_sequence(x, lengths.cpu(), batch_first=True, enforce_sorted=False)
            packed_out, _ = self.lstm(packed)
            lstm_out, _ = nn.utils.rnn.pad_packed_sequence(packed_out, batch_first=True)
        else:
            lstm_out, _ = self.lstm(x)
        return self.fc(self.dropout(lstm_out)).squeeze(-1)


class VideoDataset(Dataset):
    def __init__(self, video_ids, feature_dir, annotations):
        self.samples = []
        for vid in video_ids:
            fp = os.path.join(feature_dir, f'{vid}.npy')
            if not os.path.exists(fp): continue
            features = np.load(fp)
            scores = annotations[vid]
            n = min(len(features), len(scores))
            if n == 0: continue
            scores_norm = (scores[:n] - 1.0) / 4.0
            self.samples.append((
                torch.tensor(features[:n], dtype=torch.float32),
                torch.tensor(scores_norm, dtype=torch.float32),
            ))

    def __len__(self): return len(self.samples)
    def __getitem__(self, idx): return self.samples[idx]


def collate_fn(batch):
    features, scores = zip(*batch)
    lengths = torch.tensor([len(f) for f in features], dtype=torch.long)
    max_len = lengths.max().item()
    pf = torch.zeros(len(batch), max_len, features[0].shape[-1])
    ps = torch.zeros(len(batch), max_len)
    mask = torch.zeros(len(batch), max_len, dtype=torch.bool)
    for i, (f, s) in enumerate(zip(features, scores)):
        pf[i, :len(f)] = f
        ps[i, :len(s)] = s
        mask[i, :len(f)] = True
    return pf, ps, lengths, mask


train_set = VideoDataset(train_videos, FEATURE_DIR, annotations)
test_set = VideoDataset(test_videos, FEATURE_DIR, annotations)
train_loader = DataLoader(train_set, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_fn)
test_loader = DataLoader(test_set, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn)

model = BiLSTMSummarizer(512, HIDDEN_DIM, NUM_LAYERS, DROPOUT).to(DEVICE)
optimizer = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=1e-5)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=5, factor=0.5)
criterion = nn.MSELoss()

print(f'Parameters: {sum(p.numel() for p in model.parameters()):,}')
print(f'Train: {len(train_set)} videos, Test: {len(test_set)} videos\n')

best_val_loss = float('inf')
patience_counter = 0
history = []

for epoch in range(1, EPOCHS + 1):
    # Train
    model.train()
    train_loss = 0
    train_n = 0
    for features, scores, lengths, mask in train_loader:
        features, scores, mask = features.to(DEVICE), scores.to(DEVICE), mask.to(DEVICE)
        preds = model(features, lengths)
        loss = criterion(preds[mask], scores[mask])
        optimizer.zero_grad()
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        train_loss += loss.item() * mask.sum().item()
        train_n += mask.sum().item()
    train_loss /= max(train_n, 1)

    # Eval
    model.eval()
    val_loss = 0
    val_n = 0
    all_p, all_t = [], []
    with torch.no_grad():
        for features, scores, lengths, mask in test_loader:
            features, scores, mask = features.to(DEVICE), scores.to(DEVICE), mask.to(DEVICE)
            preds = model(features, lengths)
            loss = criterion(preds[mask], scores[mask])
            val_loss += loss.item() * mask.sum().item()
            val_n += mask.sum().item()
            all_p.append((preds[mask].cpu().numpy() * 4 + 1))
            all_t.append((scores[mask].cpu().numpy() * 4 + 1))
    val_loss /= max(val_n, 1)
    scheduler.step(val_loss)

    all_p = np.concatenate(all_p)
    all_t = np.concatenate(all_t)
    mse = mean_squared_error(all_t, all_p)
    history.append({'epoch': epoch, 'train_loss': train_loss, 'val_loss': val_loss, 'mse': mse})

    if epoch % 5 == 0 or epoch == 1:
        print(f'Epoch {epoch:3d}/{EPOCHS} | Train: {train_loss:.4f} | Val: {val_loss:.4f} | MSE: {mse:.4f}')

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        patience_counter = 0
        lstm_path = os.path.join(MODEL_DIR, 'bilstm_video_split.pt')
        torch.save(model.state_dict(), lstm_path)
    else:
        patience_counter += 1
        if patience_counter >= PATIENCE:
            print(f'\nEarly stopping at epoch {epoch}')
            break

# Final eval
model.load_state_dict(torch.load(lstm_path, map_location=DEVICE, weights_only=True))
model.eval()
all_p, all_t = [], []
with torch.no_grad():
    for features, scores, lengths, mask in test_loader:
        features, scores, mask = features.to(DEVICE), scores.to(DEVICE), mask.to(DEVICE)
        preds = model(features, lengths)
        all_p.append((preds[mask].cpu().numpy() * 4 + 1))
        all_t.append((scores[mask].cpu().numpy() * 4 + 1))
all_p, all_t = np.concatenate(all_p), np.concatenate(all_t)
lstm_mse = mean_squared_error(all_t, all_p)
lstm_mae = mean_absolute_error(all_t, all_p)
lstm_r2 = r2_score(all_t, all_p)
lstm_spearman = spearmanr(all_t, all_p).correlation

print(f'\nBiLSTM Results:')
print(f'  MSE: {lstm_mse:.4f}')
print(f'  MAE: {lstm_mae:.4f}')
print(f'  R2:  {lstm_r2:.4f}')
print(f'  Spearman: {lstm_spearman:.4f}')
print(f'Saved → {lstm_path}')

## Cell 10: Compare All Models

In [ ]:
import matplotlib.pyplot as plt

results = {
    'Model': ['Random Forest', 'XGBoost (temporal)', 'BiLSTM'],
    'MSE': [rf_mse, xgb_mse, lstm_mse],
    'MAE': [rf_mae, xgb_mae, lstm_mae],
    'R2': [rf_r2, xgb_r2, lstm_r2],
    'Spearman': [rf_spearman, xgb_spearman, lstm_spearman],
}

results_df = pd.DataFrame(results)
print('\n' + '='*70)
print('MODEL COMPARISON')
print('='*70)
print(results_df.to_string(index=False))
print('='*70)

# Determine best model
best_idx = results_df['Spearman'].idxmax()
best_model = results_df.loc[best_idx, 'Model']
print(f'\nBest model (by Spearman correlation): {best_model}')

# Plot comparison
fig, axes = plt.subplots(1, 4, figsize=(16, 4))
metrics = ['MSE', 'MAE', 'R2', 'Spearman']
colors = ['#ef4444', '#f59e0b', '#10b981', '#7c3aed']

for ax, metric, color in zip(axes, metrics, colors):
    bars = ax.bar(results_df['Model'], results_df[metric], color=color, alpha=0.8)
    ax.set_title(metric, fontweight='bold')
    ax.set_xticklabels(results_df['Model'], rotation=15, ha='right', fontsize=9)
    for bar, val in zip(bars, results_df[metric]):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01, f'{val:.3f}',
                ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.savefig(os.path.join(PROJECT_DIR, 'model_comparison.png'), dpi=150, bbox_inches='tight')
plt.show()

# Plot BiLSTM training curve
if history:
    fig, ax = plt.subplots(figsize=(10, 4))
    epochs_list = [h['epoch'] for h in history]
    ax.plot(epochs_list, [h['train_loss'] for h in history], label='Train Loss', color='#7c3aed')
    ax.plot(epochs_list, [h['val_loss'] for h in history], label='Val Loss', color='#06b6d4')
    ax.set_xlabel('Epoch')
    ax.set_ylabel('MSE Loss')
    ax.set_title('BiLSTM Training Curve')
    ax.legend()
    ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig(os.path.join(PROJECT_DIR, 'bilstm_training_curve.png'), dpi=150, bbox_inches='tight')
    plt.show()

## Cell 11: Save Best Model to Drive

Download the `outputs/models/` folder from Drive and place it in your local project.

In [ ]:
print('Models saved to Google Drive:')
print(f'  {MODEL_DIR}/')
for f in sorted(os.listdir(MODEL_DIR)):
    size_mb = os.path.getsize(os.path.join(MODEL_DIR, f)) / (1024 * 1024)
    print(f'    {f} ({size_mb:.1f} MB)')

print(f'\nTo use locally, download the models folder to:')
print(f'  your_project/outputs/models/')
print(f'\nFiles needed for inference:')
print(f'  - xgboost_video_split_temporal.pkl (Video -> Video pipeline)')
print(f'  - random_forest_video_split.pkl    (alternative model)')
print(f'  - bilstm_video_split.pt            (alternative model)')